# Analyze `predict.csv` with strict metric

Notebook nay doc file `results/predict.csv`, tinh dung/sai theo logic strict metric, va thong ke rieng cho `IE` va `IO`.

In [1]:
import re
from pathlib import Path

import pandas as pd

from utils.metrics import (
    compute_strict_metrics,
    compute_loose_metrics,
    compute_lcs_triplet_metrics,
 )

c:\Users\Ngoc\miniconda3\envs\AI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# csv_path = Path('results/vit5-base-raw(refine)-type2-m-strict.csv')
csv_path = Path('results/predict (3).csv')
df = pd.read_csv(csv_path)

print(f'Rows: {len(df):,}')
df.head(3)

Rows: 8,698


,text,predict,label
0,"Một cuốn sách hay, nhẹ nhàng và dễ thương, nhi...",[ate] cuốn sách ## cuốn sách ## Trình bày ## c...,[ate] Một cuốn sách ## nhiều câu ## nhiều câu ...
1,"Một cuốn sách hay, nhẹ nhàng và dễ thương, nhi...","[aooe] hay, nhẹ nhàng ## dễ thương, ## rất chấ...","[aooe] hay,"
2,"Một cuốn sách hay, nhẹ nhàng và dễ thương, nhi...",[aooe] có thể làm quotes rất chất ## ý nghĩa!,[aooe] rất chất ## ý nghĩa!


In [12]:
def task_from_label(label: str):
    match = re.match(r'^\s*\[(\w+)\]', str(label))
    return match.group(1).lower() if match else None


def strip_task_prefix(text: str, task: str):
    if text is None:
        return ''
    text = str(text).strip()
    prefix = f'[{task}]'

    if text.startswith(prefix):
        return text[len(prefix):].strip()
    if prefix in text:
        return text.split(prefix, 1)[1].strip()
    return text


def parse_entries(content: str, task: str):
    if not content:
        return []

    chunks = [x.strip() for x in content.split(' ## ') if x.strip()]
    if task in {'aspe', 'aope', 'aoste'}:
        return [tuple(part.strip() for part in chunk.split(' $ ')) for chunk in chunks]
    return chunks


def row_strict_true_false(predict: str, label: str):
    task = task_from_label(label)
    if not task:
        return False

    pred_content = strip_task_prefix(predict, task)
    label_content = strip_task_prefix(label, task)

    pred_entries = parse_entries(pred_content, task)
    label_entries = parse_entries(label_content, task)
    return set(pred_entries) == set(label_entries)


df['strict_correct'] = [row_strict_true_false(p, l) for p, l in zip(df['predict'], df['label'])]

pred_list = df['predict'].astype(str).tolist()
label_list = df['label'].astype(str).tolist()
lcs_threshold = 0.5

strict_metrics = compute_strict_metrics(pred_list, label_list)
loose_metrics = compute_loose_metrics(pred_list, label_list)
lcs_triplet_metrics = compute_lcs_triplet_metrics(
    pred_list,
    label_list,
    threshold=lcs_threshold,
 )

metrics_df = pd.DataFrame(
    {
        'strict': pd.Series(strict_metrics),
        'loose': pd.Series(loose_metrics),
        f'lcs_triplet@{lcs_threshold}': pd.Series(lcs_triplet_metrics),
    }
).sort_index()

display(metrics_df)

,strict,loose,lcs_triplet@0.5
aooe_f1,NaN,0.898771,0.900018
aooe_macro_f1,0.736120,NaN,NaN
aooe_macro_precision,0.740655,NaN,NaN
aooe_macro_recall,0.737452,NaN,NaN
aooe_micro_f1,0.714967,NaN,NaN
aooe_micro_precision,0.718128,NaN,NaN
aooe_micro_recall,0.711833,NaN,NaN
aooe_precision,NaN,0.890720,0.891795
aooe_recall,NaN,0.906968,0.908394
aope_f1,NaN,0.658745,0.660842


In [4]:
def subset_stats(frame: pd.DataFrame, keyword: str):
    mask = frame['label'].astype(str).str.contains(keyword, case=False, na=False)
    subset = frame.loc[mask].copy()

    summary = pd.DataFrame(
        {
            'count': subset['strict_correct'].value_counts(dropna=False)
        }
    ).rename(index={True: 'correct', False: 'wrong'}).fillna(0).astype(int)

    total = len(subset)
    summary['ratio'] = (summary['count'] / total).round(4) if total > 0 else 0.0
    return subset, summary, total


def subset_stats_without(frame: pd.DataFrame, keyword: str):
    mask = ~frame['label'].astype(str).str.contains(keyword, case=False, na=False)
    subset = frame.loc[mask].copy()

    summary = pd.DataFrame(
        {
            'count': subset['strict_correct'].value_counts(dropna=False)
        }
    ).rename(index={True: 'correct', False: 'wrong'}).fillna(0).astype(int)

    total = len(subset)
    summary['ratio'] = (summary['count'] / total).round(4) if total > 0 else 0.0
    return subset, summary, total


def is_aoste_reversible_enough(text: str):
    # AOSTE moi chunk can du 3 thanh phan: aspect $ opinion $ sentiment
    content = strip_task_prefix(text, 'aoste')
    chunks = [x.strip() for x in str(content).split(' ## ') if x.strip()]
    if not chunks:
        return False

    for chunk in chunks:
        parts = [p.strip() for p in chunk.split(' $ ') if p.strip()]
        if len(parts) < 3:
            return False
    return True


ie_subset, ie_summary, ie_total = subset_stats(df, '<IA>')
io_subset, io_summary, io_total = subset_stats(df, '<IO>')
no_ie_subset, no_ie_summary, no_ie_total = subset_stats_without(df, '<IE>')
no_io_subset, no_io_summary, no_io_total = subset_stats_without(df, '<IO>')

print(f'IE total: {ie_total}')
display(ie_summary)

print(f'IO total: {io_total}')
display(io_summary)

print(f'without IE total: {no_ie_total}')
display(no_ie_summary)

print(f'without IO total: {no_io_total}')
display(no_io_summary)

aoste_subset = df.loc[df['label'].astype(str).str.contains(r'^\\s*\\[aoste\\]', case=False, regex=True)].copy()
aoste_subset['aoste_reversible_enough'] = aoste_subset['predict'].astype(str).apply(is_aoste_reversible_enough)

aoste_reversible_summary = pd.DataFrame(
    {'count': aoste_subset['aoste_reversible_enough'].value_counts(dropna=False)}
).rename(index={True: 'enough_components', False: 'insufficient_components'}).fillna(0).astype(int)
aoste_total = len(aoste_subset)
aoste_reversible_summary['ratio'] = (aoste_reversible_summary['count'] / aoste_total).round(4) if aoste_total > 0 else 0.0

print(f'aoste total: {aoste_total}')
print('aoste reverse coverage (du 3 thanh phan aspect-opinion-sentiment):')
display(aoste_reversible_summary)

print('aoste samples with insufficient components (first 10):')
display(aoste_subset.loc[~aoste_subset['aoste_reversible_enough'], ['text', 'predict', 'label', 'strict_correct']].head(10))

IE total: 0


,count,ratio
strict_correct,,


IO total: 0


,count,ratio
strict_correct,,


without IE total: 5338


...

,count,ratio
strict_correct,,
correct,3195,0.5985
wrong,2143,0.4015


aoste total: 0
aoste reverse coverage (du 3 thanh phan aspect-opinion-sentiment):


,count,ratio
aoste_reversible_enough,,


aoste samples with insufficient components (first 10):


,text,predict,label,strict_correct


In [5]:
show_cols = ['text', 'predict', 'label', 'strict_correct']

print('IE - correct samples')
display(ie_subset.loc[ie_subset['strict_correct'], show_cols].head(10))

print('IE - wrong samples')
display(ie_subset.loc[~ie_subset['strict_correct'], show_cols].head(10))

print('IO - correct samples')
display(io_subset.loc[io_subset['strict_correct'], show_cols].head(10))

print('IO - wrong samples')
display(io_subset.loc[~io_subset['strict_correct'], show_cols].head(10))

IE - correct samples


,text,predict,label,strict_correct


IE - wrong samples


,text,predict,label,strict_correct


IO - correct samples


,text,predict,label,strict_correct


IO - wrong samples


,text,predict,label,strict_correct


In [6]:
# Thong ke do dai cau (so tu) theo nhom du doan dung/sai
length_df = df.copy()

# Tach text goc tu cot input neu cot text chua co/noi dung rong
if 'text' in length_df.columns:
    extracted_text = length_df['text'].astype(str)
else:
    extracted_text = pd.Series([''] * len(length_df), index=length_df.index)

if 'input' in length_df.columns:
    extracted_from_input = (
        length_df['input']
        .astype(str)
        .str.split('input: ', n=1)
        .str[-1]
        .str.split('\noutput:', n=1)
        .str[0]
    )
    extracted_text = extracted_text.where(extracted_text.str.strip() != '', extracted_from_input)

length_df['text_for_len'] = extracted_text.fillna('').astype(str).str.strip()
length_df['word_len'] = length_df['text_for_len'].str.split().str.len()

length_summary = (
    length_df.groupby('strict_correct')['word_len']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .rename(index={True: 'correct', False: 'wrong'})
)

print('Sentence length stats by strict correctness (word count):')
display(length_summary)

print(f"Avg length - correct: {length_df.loc[length_df['strict_correct'], 'word_len'].mean():.2f} words")
print(f"Avg length - wrong:   {length_df.loc[~length_df['strict_correct'], 'word_len'].mean():.2f} words")

Sentence length stats by strict correctness (word count):


,count,mean,median,min,max
strict_correct,,,,,
wrong,2143,34.277182,22.0,3,235
correct,3195,30.492645,21.0,3,235


Avg length - correct: 30.49 words
Avg length - wrong:   34.28 words


In [7]:
from pathlib import Path

wrong_df = df.loc[~df['strict_correct']].copy()

output_path = Path('results/predict_wrong_only.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
wrong_df.to_csv(output_path, index=False)

print(f'Saved {len(wrong_df):,} wrong samples to {output_path}')
wrong_df.head(10)

Saved 2,143 wrong samples to results\predict_wrong_only.csv


,text,predict,label,strict_correct
0,sao máy giặt tôi mua cách đây được 1 tháng. sa...,[ate] chế độ giặt ## giặt,[ate] giặt,False
1,sao máy giặt tôi mua cách đây được 1 tháng. sa...,[aooe] 1h30 có lúc giặt gần 4h mới xong.,[aooe] gần 4h mới xong.,False
2,sao máy giặt tôi mua cách đây được 1 tháng. sa...,[aope] máy giặt $ no_opinion ## giặt $ gần 4h ...,[aope] giặt $ gần 4h mới xong.,False
4,sao máy giặt tôi mua cách đây được 1 tháng. sa...,[aspe] máy giặt $ NEG ## giặt $ NEG ## chế độ ...,[aspe] giặt $ NEG,False
5,sao máy giặt tôi mua cách đây được 1 tháng. sa...,[aoste] giặt $ gần 4h mới xong. $ NEG ## chế đ...,[aoste] giặt $ gần 4h mới xong. $ NEG,False
17,Toi rat chi la hai long cm on shop toi se ung ...,[aooe] rat chi la hai long ## se ung ho them,[aooe] hai long ## se ung ho,False
18,Toi rat chi la hai long cm on shop toi se ung ...,[aooe] se ung ho them,[aooe] cm on,False
19,Toi rat chi la hai long cm on shop toi se ung ...,[aope] implicit_aspect $ rat chi la hai long #...,[aope] implicit_aspect $ hai long ## shop $ cm...,False
22,Toi rat chi la hai long cm on shop toi se ung ...,[aspe] implicit_aspect_0 $ POS ## shop $ POS #...,[aspe] implicit_aspect_0 $ POS ## shop $ POS #...,False
23,Toi rat chi la hai long cm on shop toi se ung ...,[aoste] implicit_aspect $ rat chi la hai long ...,[aoste] implicit_aspect $ hai long $ POS ## sh...,False
